# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook provides a walkthrough for loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library. The dataset contains survey data on mental health indicators among residents of Kilifi County, including demographic details and measurements from validated psychological assessments (GAD-7, PHQ-9, PSQ).

### Dataset Source
This dataset is described using a [Croissant schema](https://mlcommons.org/croissant/spec/) accessible via the following URL:

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and explore the records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
# Display basic information
print(f"Dataset: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Dataset identifier: {getattr(metadata, 'identifier', None)}")

# Output available record sets
if hasattr(metadata, 'record_sets') and metadata.record_sets:
    record_set_ids = [r['@id'] for r in metadata.record_sets]
    print(f"Available record sets: {record_set_ids}")
else:
    print("No explicit record sets listed in top-level metadata. Attempting autodetect...")

# In later steps, we will enumerate all discovered record sets from the Croissant graph/metadata.

## 2. Data Overview
Review available record sets, fields, and their `@id`s. This helps you identify which record sets and fields you may want to analyze in detail.

**Note:** In Croissant, every entity (record set, field, column) has an `@id` for reference.

In [ ]:
# List all record sets in the Croissant metadata
# These may be available as metadata.record_sets or as part of the graph structure.

def list_record_sets(ds_metadata):
    # Try (1) .record_sets attribute, (2) fallback by scanning the expanded JSON-LD
    ids = set()
    # Case 1: direct
    if hasattr(ds_metadata, 'record_sets') and ds_metadata.record_sets:
        for rec in ds_metadata.record_sets:
            ids.add(rec['@id'])
    
    # Case 2: expanded
    # Try to access the expanded graph or JSON-LD
    try:
        as_json = ds_metadata.to_jsonld() if hasattr(ds_metadata, 'to_jsonld') else ds_metadata.to_json()
        graph = as_json.get('@graph', [])
        for ent in graph:
            if ent.get('@type', '') in ['cr:RecordSet', 'RecordSet']:
                ids.add(ent['@id'])
    except Exception:
        pass
    return sorted(ids)

record_set_ids = list_record_sets(metadata)
if not record_set_ids:
    print("No record sets detected in metadata. Check the Croissant source.")
else:
    print("Record sets found:")
    for rid in record_set_ids:
        print(f" - {rid}")

# List all available fields for each record set (by their @id)
# We display the first one as example
if record_set_ids:
    first_rs = record_set_ids[0]
    print(f"\nExploring fields for record set: {first_rs}\n")
    # Try to find field IDs using the Croissant metadata
    rs_fields = []
    try:
        # Get the JSON-LD graph
        croissant_json = metadata.to_json() if hasattr(metadata, 'to_json') else metadata
        graph = croissant_json.get('@graph', [])
        for entity in graph:
            if entity.get('@id') == first_rs:
                # Croissant 1.0 convention: 'field' or 'fields' as a list of objects
                _fields = entity.get('field', []) or entity.get('fields', [])
                for f in _fields:
                    if isinstance(f, dict) and '@id' in f:
                        rs_fields.append(f['@id'])
                    elif isinstance(f, str):
                        rs_fields.append(f)
    except Exception as e:
        print(f"Exception while searching for fields: {e}")
    if rs_fields:
        print(f"Fields in {first_rs}:\n" + "\n".join([f" - {fid}" for fid in rs_fields]))
    else:
        print("No fields found for the selected record set.")

# Print the first few records of the selected record set
print(f"\nSample records from {first_rs}:")
cnt = 0
for record in dataset.records(record_set=first_rs):
    pprint(record)
    cnt += 1
    if cnt >= 2:
        break

## 3. Data Extraction
Load data from record sets into pandas DataFrames for analysis. Reference both the record set and field `@id`s found above.

Depending on the dataset, there may be multiple record sets; we load each as a separate DataFrame for analysis.

In [ ]:
# Extract data from all detected record sets
import warnings

dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records from record set: {rs_id}")
    try:
        # Collect all records as list of dicts
        records = list(dataset.records(record_set=rs_id))
        if records:
            # If rows are not all dicts, coerce to dict
            records = [{**r} for r in records if isinstance(r, dict)]
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            # Show basic info
            print(f"Columns in {rs_id}: {df.columns.tolist()}")
            print(df.head(2))
        else:
            print(f"No records found in record set {rs_id}.")
    except Exception as e:
        warnings.warn(f"Failed to load record set {rs_id}: {e}")

# Pick the main record set for later analysis (default to the first)
main_record_set_id = record_set_ids[0] if record_set_ids else None
main_df = dataframes[main_record_set_id] if main_record_set_id in dataframes else None

## 4. Exploratory Data Analysis (EDA)
Now, we demonstrate basic data processing:
- Filtering records based on a numeric field (e.g. GAD-7 Score)
- Normalizing values
- Grouping by categorical variables

We will reference each field solely by its `@id`, per Croissant best practices. Please substitute with the actual `@id` from above if you want to select a different field.

In [ ]:
if main_df is not None and not main_df.empty:
    # Choose an example numerical field (using its @id)
    print("Available columns in main data frame:")
    print(main_df.columns.tolist())
    # For the purpose of this notebook, we'll try common mental health measure names.
    # Update these with the actual @id from section 2 if different.
    gad7_id = None
    for c in main_df.columns:
        if 'gad7' in c.lower():
            gad7_id = c
            break
    if not gad7_id:
        # fallback to first numeric column
        for col in main_df.select_dtypes(include='number').columns:
            gad7_id = col
            break
    print(f"Numeric field for analysis: {gad7_id}")
    if gad7_id is not None:
        threshold = 10
        filtered_df = main_df[main_df[gad7_id] > threshold].copy()
        print(f"Filtered records with {gad7_id} > {threshold}:")
        print(filtered_df.head())
        # Normalization
        filtered_df[f"{gad7_id}_normalized"] = (filtered_df[gad7_id] - filtered_df[gad7_id].mean()) / filtered_df[gad7_id].std()
        print(f"Normalized {gad7_id}:")
        print(filtered_df[[gad7_id, f"{gad7_id}_normalized"].copy()].head())
        # Group by a demographic field
        group_field = None
        for col in main_df.columns:
            if any(keyword in col.lower() for keyword in ['gender', 'sex', 'village', 'education']):
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped by {group_field} (mean values):")
            print(grouped_df[[gad7_id, f"{gad7_id}_normalized"]].head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric field found in dataset for filtering/normalization.")
else:
    print("No main DataFrame loaded or DataFrame is empty.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here, we show the distribution of the chosen numeric score, grouped by a demographic variable (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_df is not None and not main_df.empty and gad7_id is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(main_df[gad7_id].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of {gad7_id} scores")
    plt.xlabel(gad7_id)
    plt.show()
    
    # If grouping field available
    if group_field is not None:
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=main_df, x=group_field, y=gad7_id)
        plt.title(f"{gad7_id} score by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(gad7_id)
        plt.xticks(rotation=30, ha='right')
        plt.show()

## 6. Conclusion
- We demonstrated loading and exploring the FAIR² Mental Health Survey Dataset using `mlcroissant` based on its Croissant schema.
- Record sets and field `@id`s were discovered and referenced throughout as required for best interoperability.
- We performed basic EDA, filtering, normalization, grouping by demographic variable, and visualization.
- This approach can be extended for deeper statistical analysis, model building, or integration with other Croissant-conformant resources.

Refer to the dataset's Croissant schema for further available fields and analytic opportunities.